# 1. 개요

본 보고서는 1차 도메인 프로젝트에서 사용되는 **Machine Reading Comprehension(MRC)** 데이터의 품질을 검증하고, 모델 학습 및 추론 과정에서 발생할 수 있는 오류를 최소화하기 위한 목적으로 작성되었다.

프로젝트 데이터는 다음 두 가지로 구성된다.

1. **MRC Q&A 데이터셋(train/validation/test)**

2. **Retrieval을 위한 corpus(wikipedia_documents.json)**

MRC 모델은 질문(question)에 대해 context에서 정답을 찾아내는 구조이며, context는 corpus 문서 집합에서 retrieval 모델을 통해 검색된다. 따라서 **MRC 데이터와 corpus는 상호 의존적인 구조**를 가지며, 두 데이터 모두에 대한 품질 검사가 필수적이다.


# 2. 데이터 구성


## 2.1 MRC 데이터셋 구조


In [ ]:
# 라이브러리 임포트
from datasets import load_from_disk
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("라이브러리 임포트 완료")


In [ ]:
# MRC 데이터셋 로드
print("MRC 데이터셋 로드 중...")
train_dataset = load_from_disk("../../data/train_dataset")

train_df = pd.DataFrame(train_dataset["train"])
val_df = pd.DataFrame(train_dataset["validation"])

print(f"Train 데이터: {len(train_df)}개")
print(f"Validation 데이터: {len(val_df)}개")
print(f"\n컬럼: {train_df.columns.tolist()}")
print(f"\n샘플 데이터:")
train_df.head(3)


In [ ]:
# MRC 데이터셋 구조 상세 분석
print("=" * 60)
print("MRC 데이터셋 구조 분석")
print("=" * 60)

# 각 필드의 데이터 타입 및 샘플
print("\n[1] ID 필드")
print(f"  - 타입: {type(train_df['id'].iloc[0])}")
print(f"  - 샘플: {train_df['id'].iloc[0]}")

print("\n[2] Question 필드")
print(f"  - 타입: {type(train_df['question'].iloc[0])}")
print(f"  - 샘플: {train_df['question'].iloc[0][:100]}...")

print("\n[3] Context 필드")
print(f"  - 타입: {type(train_df['context'].iloc[0])}")
print(f"  - 샘플: {train_df['context'].iloc[0][:200]}...")

print("\n[4] Answers 필드")
print(f"  - 타입: {type(train_df['answers'].iloc[0])}")
print(f"  - 구조: {train_df['answers'].iloc[0]}")
print(f"  - Keys: {train_df['answers'].iloc[0].keys() if isinstance(train_df['answers'].iloc[0], dict) else 'N/A'}")

print("\n[5] Title 필드")
print(f"  - 타입: {type(train_df['title'].iloc[0])}")
print(f"  - 샘플: {train_df['title'].iloc[0]}")

print("\n[6] Document ID 필드")
print(f"  - 타입: {type(train_df['document_id'].iloc[0])}")
print(f"  - 샘플: {train_df['document_id'].iloc[0]}")


## 2.2 Corpus(wikipedia_document.json) 데이터 구조


In [ ]:
# Corpus 데이터 로드
corpus_path = "../../data/wikipedia_documents.json"

try:
    with open(corpus_path, "r", encoding="utf-8") as f:
        corpus_data = json.load(f)
    
    print("=" * 60)
    print("Corpus 데이터 구조 분석")
    print("=" * 60)
    print(f"\n총 문서 개수: {len(corpus_data)}")
    
    # 첫 번째 문서 구조 확인
    first_key = list(corpus_data.keys())[0]
    first_doc = corpus_data[first_key]
    
    print(f"\n문서 키 형식: {first_key}")
    print(f"문서 구조: {list(first_doc.keys())}")
    print(f"\n샘플 문서:")
    print(f"  - text 길이: {len(first_doc.get('text', ''))}자")
    print(f"  - text 미리보기: {first_doc.get('text', '')[:200]}...")
    
    # 모든 문서의 필드 확인
    all_keys = set()
    for doc in corpus_data.values():
        if isinstance(doc, dict):
            all_keys.update(doc.keys())
    
    print(f"\n문서에 포함된 모든 필드: {sorted(all_keys)}")
    
except FileNotFoundError:
    print(f"경고: {corpus_path} 파일을 찾을 수 없습니다.")
    corpus_data = None
except Exception as e:
    print(f"오류 발생: {e}")
    corpus_data = None


# 3. 점검 항목


## 3.1 MRC 데이터셋 점검 항목

다음 항목들을 점검합니다:

1. **기본 무결성 검사**
   - 중복 ID 확인
   - 필수 필드 누락 확인

2. **Answer 검증**
   - answers["text"]가 빈 문자열인 경우
   - answer_start가 음수이거나 범위를 벗어나는 경우
   - answer_start 위치의 텍스트가 실제 answer text와 일치하는지 (annotation mismatch)
   - answer_start + answer_text 길이가 context 길이를 초과하는지
   - Answer Text 앞뒤 공백 확인

3. **Context 검증**
   - context가 비어 있거나 너무 짧은 경우
   - Context 길이 이상치 확인 (IQR 방식)
   - HTML 태그/Markup 포함 여부 확인

4. **Question 검증**
   - Question 길이 이상치 확인

5. **데이터 일관성 검사**
   - 정답이 context 안에 여러 번 등장하는 경우 확인
   - 중복된 (question, context) 쌍 확인
   - Title과 Context 불일치 확인
   - 공백/개행 문자로 인한 mismatch 검사
   - answer_start 오프셋 오류 확인 (±1 위치)


## 3.2 Corpus 데이터 점검 항목

다음 항목들을 점검합니다:

1. **기본 통계**
   - 총 문서 개수
   - 문서 길이 분포
   - 평균/중앙값/최소/최대 길이

2. **데이터 무결성**
   - 중복 문서 존재 여부
   - 빈 문서 확인
   - 문서 키 중복 확인

3. **데이터 품질**
   - HTML 태그/Markup 포함 여부
   - 특수 문자 포함 여부
   - 인코딩 문제 확인


# 4. 점검 결과


## 4.1 MRC 데이터셋 품질 결과


In [ ]:
# MRC 데이터셋 품질 점검 함수 정의

def check_duplicate_ids(df):
    """중복 ID 확인"""
    duplicates = df[df.duplicated(subset=['id'], keep=False)]
    return duplicates

def check_empty_answers(df):
    """빈 answer 확인"""
    empty_indices = []
    for idx, row in df.iterrows():
        answers = row['answers']
        if isinstance(answers, dict) and 'text' in answers:
            texts = answers['text']
            if isinstance(texts, list):
                if any(text == '' or text is None for text in texts):
                    empty_indices.append(idx)
    return empty_indices

def check_context_length(df):
    """Context 길이 확인"""
    df['context_length'] = df['context'].apply(len)
    empty = df[df['context_length'] == 0]
    short = df[df['context_length'] < 10]
    return empty, short, df

def check_answer_start_range(df):
    """answer_start 범위 확인"""
    negative_indices = []
    out_of_range_indices = []
    
    for idx, row in df.iterrows():
        answers = row['answers']
        context = row['context']
        context_len = len(context)
        
        if isinstance(answers, dict) and 'answer_start' in answers:
            starts = answers['answer_start']
            if isinstance(starts, list):
                for start in starts:
                    if start < 0:
                        negative_indices.append(idx)
                    elif start >= context_len:
                        out_of_range_indices.append(idx)
    
    return negative_indices, out_of_range_indices

def check_annotation_mismatch(df):
    """Annotation 불일치 확인"""
    mismatch_indices = []
    
    for idx, row in df.iterrows():
        answers = row['answers']
        context = row['context']
        
        if isinstance(answers, dict) and 'answer_start' in answers and 'text' in answers:
            starts = answers['answer_start']
            texts = answers['text']
            
            if isinstance(starts, list) and isinstance(texts, list):
                for start, text in zip(starts, texts):
                    if start < 0 or start >= len(context):
                        continue
                    extracted_text = context[start:start+len(text)]
                    if extracted_text != text:
                        mismatch_indices.append(idx)
                        break
    
    return mismatch_indices

def check_answer_span_out_of_range(df):
    """Answer span 범위 초과 확인"""
    out_of_range_indices = []
    
    for idx, row in df.iterrows():
        answers = row['answers']
        context = row['context']
        context_len = len(context)
        
        if isinstance(answers, dict) and 'answer_start' in answers and 'text' in answers:
            starts = answers['answer_start']
            texts = answers['text']
            
            if isinstance(starts, list) and isinstance(texts, list):
                for start, text in zip(starts, texts):
                    if start < 0:
                        continue
                    answer_end = start + len(text)
                    if answer_end > context_len:
                        out_of_range_indices.append(idx)
                        break
    
    return out_of_range_indices

def check_duplicate_answer_occurrences(df):
    """정답 중복 등장 확인"""
    duplicate_indices = []
    
    for idx, row in df.iterrows():
        answers = row['answers']
        context = row['context']
        
        if isinstance(answers, dict) and 'answer_start' in answers and 'text' in answers:
            starts = answers['answer_start']
            texts = answers['text']
            
            if isinstance(starts, list) and isinstance(texts, list):
                for start, text in zip(starts, texts):
                    if start < 0 or start >= len(context):
                        continue
                    occurrences = []
                    search_start = 0
                    while True:
                        pos = context.find(text, search_start)
                        if pos == -1:
                            break
                        occurrences.append(pos)
                        search_start = pos + 1
                    
                    if len(occurrences) > 1:
                        duplicate_indices.append(idx)
                        break
    
    return duplicate_indices

def detect_outliers_iqr(df, column):
    """IQR 방식으로 이상치 탐지"""
    values = df[column].values
    Q1 = np.percentile(values, 25)
    Q3 = np.percentile(values, 75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers, lower_bound, upper_bound, Q1, Q3, IQR

def check_html_markup(df):
    """HTML 태그/Markup 포함 확인"""
    import re
    html_pattern = re.compile(r'<[^>]+>')
    markup_pattern = re.compile(r'&[a-z]+;|&#\d+;')
    
    markup_indices = []
    for idx, row in df.iterrows():
        context = str(row['context'])
        html_tags = html_pattern.findall(context)
        markup_entities = markup_pattern.findall(context)
        if html_tags or markup_entities:
            markup_indices.append(idx)
    
    return markup_indices

def check_title_context_mismatch(df):
    """Title과 Context 불일치 확인"""
    mismatch_indices = []
    
    for idx, row in df.iterrows():
        title = str(row['title']) if 'title' in row else ''
        context = str(row['context'])
        
        if not title or not context:
            continue
        
        title_words = set(title.split())
        context_words = set(context.split())
        
        if len(title_words) > 0:
            common_words = title_words & context_words
            overlap_ratio = len(common_words) / len(title_words)
            
            if overlap_ratio < 0.1 and len(title_words) > 2:
                mismatch_indices.append(idx)
    
    return mismatch_indices

def check_duplicate_question_context_pairs(df):
    """중복된 (question, context) 쌍 확인"""
    df_copy = df.copy()
    df_copy['question_context_pair'] = df_copy['question'] + '|||' + df_copy['context']
    duplicates = df_copy[df_copy.duplicated(subset=['question_context_pair'], keep=False)]
    return duplicates

print("품질 점검 함수 정의 완료")


In [ ]:
# Train 데이터 품질 점검 실행
print("=" * 60)
print("Train 데이터 품질 점검")
print("=" * 60)

train_results = {}

# 1. 중복 ID 확인
train_duplicate_ids = check_duplicate_ids(train_df)
train_results['중복 ID'] = len(train_duplicate_ids)
print(f"\n[1] 중복 ID: {len(train_duplicate_ids)}개")

# 2. 빈 answer 확인
train_empty_answers = check_empty_answers(train_df)
train_results['빈 answer'] = len(train_empty_answers)
print(f"[2] 빈 answer: {len(train_empty_answers)}개")

# 3. Context 길이 확인
train_empty_context, train_short_context, train_df = check_context_length(train_df)
train_results['빈 context'] = len(train_empty_context)
train_results['짧은 context'] = len(train_short_context)
print(f"[3] 빈 context: {len(train_empty_context)}개, 짧은 context: {len(train_short_context)}개")
print(f"    평균 context 길이: {train_df['context_length'].mean():.2f}자")

# 4. answer_start 범위 확인
train_neg_start, train_out_range = check_answer_start_range(train_df)
train_results['음수 answer_start'] = len(train_neg_start)
train_results['범위 초과 answer_start'] = len(train_out_range)
print(f"[4] 음수 answer_start: {len(train_neg_start)}개, 범위 초과: {len(train_out_range)}개")

# 5. Annotation 불일치 확인
train_mismatch = check_annotation_mismatch(train_df)
train_results['Annotation 불일치'] = len(train_mismatch)
print(f"[5] Annotation 불일치: {len(train_mismatch)}개")

# 6. Answer span 범위 초과 확인
train_span_out = check_answer_span_out_of_range(train_df)
train_results['Span 범위 초과'] = len(train_span_out)
print(f"[6] Span 범위 초과: {len(train_span_out)}개")

# 7. 정답 중복 등장 확인
train_dup_ans = check_duplicate_answer_occurrences(train_df)
train_results['정답 중복 등장'] = len(train_dup_ans)
print(f"[7] 정답 중복 등장: {len(train_dup_ans)}개")

# 8. Context 이상치 확인
train_outliers, _, _, _, _, _ = detect_outliers_iqr(train_df, 'context_length')
train_results['Context 이상치'] = len(train_outliers)
print(f"[8] Context 이상치: {len(train_outliers)}개 ({len(train_outliers)/len(train_df)*100:.2f}%)")

# 9. Question 길이 확인
train_df['question_length'] = train_df['question'].apply(len)
train_long_questions = train_df[train_df['question_length'] > 500]
train_results['긴 question'] = len(train_long_questions)
print(f"[9] 긴 question (500자 초과): {len(train_long_questions)}개")

# 10. 중복 (Q, C) 쌍 확인
train_dup_pairs = check_duplicate_question_context_pairs(train_df)
train_results['중복 (Q,C) 쌍'] = len(train_dup_pairs)
print(f"[10] 중복 (Q,C) 쌍: {len(train_dup_pairs)}개")

# 11. Title-Context 불일치 확인
train_title_mismatch = check_title_context_mismatch(train_df)
train_results['Title-Context 불일치'] = len(train_title_mismatch)
print(f"[11] Title-Context 불일치: {len(train_title_mismatch)}개")

# 12. HTML/Markup 포함 확인
train_markup = check_html_markup(train_df)
train_results['HTML/Markup 포함'] = len(train_markup)
print(f"[12] HTML/Markup 포함: {len(train_markup)}개")

print("\n" + "=" * 60)
print("Train 데이터 품질 점검 완료")
print("=" * 60)


In [ ]:
# Validation 데이터 품질 점검 실행
print("=" * 60)
print("Validation 데이터 품질 점검")
print("=" * 60)

val_results = {}

# 1. 중복 ID 확인
val_duplicate_ids = check_duplicate_ids(val_df)
val_results['중복 ID'] = len(val_duplicate_ids)
print(f"\n[1] 중복 ID: {len(val_duplicate_ids)}개")

# 2. 빈 answer 확인
val_empty_answers = check_empty_answers(val_df)
val_results['빈 answer'] = len(val_empty_answers)
print(f"[2] 빈 answer: {len(val_empty_answers)}개")

# 3. Context 길이 확인
val_empty_context, val_short_context, val_df = check_context_length(val_df)
val_results['빈 context'] = len(val_empty_context)
val_results['짧은 context'] = len(val_short_context)
print(f"[3] 빈 context: {len(val_empty_context)}개, 짧은 context: {len(val_short_context)}개")
print(f"    평균 context 길이: {val_df['context_length'].mean():.2f}자")

# 4. answer_start 범위 확인
val_neg_start, val_out_range = check_answer_start_range(val_df)
val_results['음수 answer_start'] = len(val_neg_start)
val_results['범위 초과 answer_start'] = len(val_out_range)
print(f"[4] 음수 answer_start: {len(val_neg_start)}개, 범위 초과: {len(val_out_range)}개")

# 5. Annotation 불일치 확인
val_mismatch = check_annotation_mismatch(val_df)
val_results['Annotation 불일치'] = len(val_mismatch)
print(f"[5] Annotation 불일치: {len(val_mismatch)}개")

# 6. Answer span 범위 초과 확인
val_span_out = check_answer_span_out_of_range(val_df)
val_results['Span 범위 초과'] = len(val_span_out)
print(f"[6] Span 범위 초과: {len(val_span_out)}개")

# 7. 정답 중복 등장 확인
val_dup_ans = check_duplicate_answer_occurrences(val_df)
val_results['정답 중복 등장'] = len(val_dup_ans)
print(f"[7] 정답 중복 등장: {len(val_dup_ans)}개")

# 8. Context 이상치 확인
val_outliers, _, _, _, _, _ = detect_outliers_iqr(val_df, 'context_length')
val_results['Context 이상치'] = len(val_outliers)
print(f"[8] Context 이상치: {len(val_outliers)}개 ({len(val_outliers)/len(val_df)*100:.2f}%)")

# 9. Question 길이 확인
val_df['question_length'] = val_df['question'].apply(len)
val_long_questions = val_df[val_df['question_length'] > 500]
val_results['긴 question'] = len(val_long_questions)
print(f"[9] 긴 question (500자 초과): {len(val_long_questions)}개")

# 10. 중복 (Q, C) 쌍 확인
val_dup_pairs = check_duplicate_question_context_pairs(val_df)
val_results['중복 (Q,C) 쌍'] = len(val_dup_pairs)
print(f"[10] 중복 (Q,C) 쌍: {len(val_dup_pairs)}개")

# 11. Title-Context 불일치 확인
val_title_mismatch = check_title_context_mismatch(val_df)
val_results['Title-Context 불일치'] = len(val_title_mismatch)
print(f"[11] Title-Context 불일치: {len(val_title_mismatch)}개")

# 12. HTML/Markup 포함 확인
val_markup = check_html_markup(val_df)
val_results['HTML/Markup 포함'] = len(val_markup)
print(f"[12] HTML/Markup 포함: {len(val_markup)}개")

print("\n" + "=" * 60)
print("Validation 데이터 품질 점검 완료")
print("=" * 60)


In [ ]:
# MRC 데이터셋 품질 점검 결과 요약
print("=" * 60)
print("MRC 데이터셋 품질 점검 결과 요약")
print("=" * 60)

summary_df = pd.DataFrame({
    'Train': train_results,
    'Validation': val_results
})

# 총 개수 추가
summary_df.loc['총 개수'] = [len(train_df), len(val_df)]

# 비율 계산
summary_df['Train 비율(%)'] = (summary_df['Train'] / len(train_df) * 100).round(2)
summary_df['Validation 비율(%)'] = (summary_df['Validation'] / len(val_df) * 100).round(2)

print("\n점검 항목별 결과:")
print(summary_df.to_string())

# 시각화: 주요 문제 항목
problem_items = ['정답 중복 등장', 'Context 이상치', 'Title-Context 불일치', 'HTML/Markup 포함']
problem_data = summary_df.loc[problem_items, ['Train', 'Validation']]

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(problem_items))
width = 0.35

ax.bar(x - width/2, problem_data['Train'], width, label='Train', alpha=0.8)
ax.bar(x + width/2, problem_data['Validation'], width, label='Validation', alpha=0.8)

ax.set_xlabel('점검 항목')
ax.set_ylabel('문제 개수')
ax.set_title('MRC 데이터셋 주요 문제 항목 비교')
ax.set_xticks(x)
ax.set_xticklabels(problem_items, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Context 및 Question 길이 분포 시각화
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Train Context 길이 분포
axes[0, 0].hist(train_df['context_length'], bins=50, alpha=0.7, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Train Context 길이 분포')
axes[0, 0].set_xlabel('Context 길이 (자)')
axes[0, 0].set_ylabel('빈도')
axes[0, 0].axvline(train_df['context_length'].mean(), color='red', linestyle='--', label=f'평균: {train_df["context_length"].mean():.0f}자')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Validation Context 길이 분포
axes[0, 1].hist(val_df['context_length'], bins=50, alpha=0.7, color='lightcoral', edgecolor='black')
axes[0, 1].set_title('Validation Context 길이 분포')
axes[0, 1].set_xlabel('Context 길이 (자)')
axes[0, 1].set_ylabel('빈도')
axes[0, 1].axvline(val_df['context_length'].mean(), color='red', linestyle='--', label=f'평균: {val_df["context_length"].mean():.0f}자')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Train Question 길이 분포
axes[1, 0].hist(train_df['question_length'], bins=30, alpha=0.7, color='lightgreen', edgecolor='black')
axes[1, 0].set_title('Train Question 길이 분포')
axes[1, 0].set_xlabel('Question 길이 (자)')
axes[1, 0].set_ylabel('빈도')
axes[1, 0].axvline(train_df['question_length'].mean(), color='red', linestyle='--', label=f'평균: {train_df["question_length"].mean():.1f}자')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Validation Question 길이 분포
axes[1, 1].hist(val_df['question_length'], bins=30, alpha=0.7, color='plum', edgecolor='black')
axes[1, 1].set_title('Validation Question 길이 분포')
axes[1, 1].set_xlabel('Question 길이 (자)')
axes[1, 1].set_ylabel('빈도')
axes[1, 1].axvline(val_df['question_length'].mean(), color='red', linestyle='--', label=f'평균: {val_df["question_length"].mean():.1f}자')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 4.2 Corpus 품질 결과


In [ ]:
# Corpus 데이터 품질 점검
if corpus_data is not None:
    print("=" * 60)
    print("Corpus 데이터 품질 점검")
    print("=" * 60)
    
    # 문서 텍스트 추출
    corpus_texts = []
    corpus_lengths = []
    empty_docs = []
    
    for key, doc in corpus_data.items():
        if isinstance(doc, dict):
            text = doc.get('text', '')
            corpus_texts.append(text)
            length = len(text)
            corpus_lengths.append(length)
            if length == 0:
                empty_docs.append(key)
    
    corpus_lengths = np.array(corpus_lengths)
    
    print(f"\n[1] 기본 통계")
    print(f"  - 총 문서 개수: {len(corpus_data)}")
    print(f"  - 평균 문서 길이: {corpus_lengths.mean():.2f}자")
    print(f"  - 중앙값 문서 길이: {np.median(corpus_lengths):.2f}자")
    print(f"  - 최소 문서 길이: {corpus_lengths.min()}자")
    print(f"  - 최대 문서 길이: {corpus_lengths.max()}자")
    print(f"  - 표준편차: {corpus_lengths.std():.2f}자")
    
    print(f"\n[2] 빈 문서 확인")
    print(f"  - 빈 문서 개수: {len(empty_docs)}개")
    if len(empty_docs) > 0:
        print(f"  - 빈 문서 샘플: {empty_docs[:5]}")
    
    # 중복 문서 확인
    print(f"\n[3] 중복 문서 확인")
    unique_texts = list(dict.fromkeys(corpus_texts))
    duplicate_count = len(corpus_texts) - len(unique_texts)
    print(f"  - 고유 문서 개수: {len(unique_texts)}개")
    print(f"  - 중복 문서 개수: {duplicate_count}개")
    print(f"  - 중복률: {duplicate_count/len(corpus_texts)*100:.2f}%")
    
    # HTML/Markup 확인
    print(f"\n[4] HTML/Markup 포함 확인")
    import re
    html_pattern = re.compile(r'<[^>]+>')
    markup_pattern = re.compile(r'&[a-z]+;|&#\d+;')
    
    markup_count = 0
    for text in corpus_texts:
        if html_pattern.findall(text) or markup_pattern.findall(text):
            markup_count += 1
    
    print(f"  - Markup 포함 문서: {markup_count}개 ({markup_count/len(corpus_texts)*100:.2f}%)")
    
    # 이상치 확인 (IQR)
    print(f"\n[5] 문서 길이 이상치 확인 (IQR)")
    Q1 = np.percentile(corpus_lengths, 25)
    Q3 = np.percentile(corpus_lengths, 75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = corpus_lengths[(corpus_lengths < lower_bound) | (corpus_lengths > upper_bound)]
    print(f"  - Q1: {Q1:.0f}자, Q3: {Q3:.0f}자, IQR: {IQR:.0f}자")
    print(f"  - 정상 범위: {lower_bound:.0f} ~ {upper_bound:.0f}자")
    print(f"  - 이상치 개수: {len(outliers)}개 ({len(outliers)/len(corpus_lengths)*100:.2f}%)")
    
    # 문서 길이 분포 시각화
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(corpus_lengths, bins=100, alpha=0.7, color='steelblue', edgecolor='black')
    axes[0].set_title('Corpus 문서 길이 분포')
    axes[0].set_xlabel('문서 길이 (자)')
    axes[0].set_ylabel('빈도')
    axes[0].axvline(corpus_lengths.mean(), color='red', linestyle='--', label=f'평균: {corpus_lengths.mean():.0f}자')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # 로그 스케일
    axes[1].hist(corpus_lengths, bins=100, alpha=0.7, color='steelblue', edgecolor='black')
    axes[1].set_title('Corpus 문서 길이 분포 (로그 스케일)')
    axes[1].set_xlabel('문서 길이 (자)')
    axes[1].set_ylabel('빈도')
    axes[1].set_yscale('log')
    axes[1].axvline(corpus_lengths.mean(), color='red', linestyle='--', label=f'평균: {corpus_lengths.mean():.0f}자')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
else:
    print("Corpus 데이터를 로드할 수 없어 품질 점검을 수행할 수 없습니다.")


# 5. 향후 개선 방안

## 5.1 MRC 데이터셋 개선 방안

### 발견된 주요 문제점

1. **정답 중복 등장**
   - Train: 1,312개 (33.2%), Validation: 78개 (32.5%)
   - **개선 방안**: 
     - 정답이 여러 번 등장하는 경우, 가장 적절한 위치를 선택하거나 모든 위치를 정답으로 인정
     - 모델 학습 시 여러 정답 위치를 모두 고려하도록 데이터 전처리

2. **Context 이상치**
   - Train: 155개 (3.92%), Validation: 7개 (2.92%)
   - **개선 방안**:
     - 매우 긴 context는 적절히 분할하거나 요약
     - 모델의 최대 입력 길이를 고려한 전처리

3. **Title-Context 불일치**
   - Train: 269개 (6.81%), Validation: 17개 (7.08%)
   - **개선 방안**:
     - Title과 Context의 관련성을 더 정확히 검증
     - 불일치가 의심되는 경우 수동 검토

4. **HTML/Markup 포함**
   - Train: 91개 (2.30%), Validation: 3개 (1.25%)
   - **개선 방안**:
     - HTML 태그 제거 또는 정규화
     - Markup 엔티티 디코딩

## 5.2 Corpus 데이터 개선 방안

### 발견된 주요 문제점

1. **중복 문서**
   - **개선 방안**: 중복 문서 제거 또는 통합

2. **문서 길이 편차**
   - **개선 방안**: 
     - 너무 짧거나 긴 문서에 대한 처리 기준 수립
     - Retrieval 성능에 영향을 주는 문서 길이 범위 설정

3. **Markup 포함**
   - **개선 방안**: HTML 태그 및 특수 문자 정제


# 6. 결론

본 데이터 품질 검증을 통해 다음과 같은 결론을 도출할 수 있다:

## 6.1 MRC 데이터셋 품질 평가

**전반적인 품질**: 양호

- ✅ **기본 무결성**: 중복 ID, 빈 answer, 범위 초과 등 심각한 오류 없음
- ✅ **Annotation 정확성**: answer_start와 answer_text의 일치도 높음
- ⚠️ **주의 필요 항목**: 
  - 정답 중복 등장 (약 33%)
  - Title-Context 불일치 (약 7%)
  - HTML/Markup 포함 (약 2%)

## 6.2 Corpus 데이터 품질 평가

**전반적인 품질**: 양호

- ✅ **문서 구조**: 대부분의 문서가 정상적인 구조를 가짐
- ⚠️ **주의 필요 항목**:
  - 중복 문서 존재 가능성
  - 문서 길이 편차

## 6.3 권장 사항

1. **즉시 조치 필요**: 없음 (심각한 오류 없음)

2. **단기 개선 사항**:
   - HTML/Markup 정제
   - Title-Context 불일치 샘플 검토

3. **장기 개선 사항**:
   - 정답 중복 등장 처리 방안 수립
   - Corpus 중복 문서 제거
   - 문서 길이 최적화

4. **모델 학습 시 고려사항**:
   - 정답이 여러 번 등장하는 경우를 고려한 학습 전략
   - Context 길이에 따른 적절한 truncation 전략
   - Title 정보를 활용한 추가 feature 고려

---

**보고서 작성일**: 2024년  
**데이터 버전**: 1차 도메인 프로젝트 데이터셋
